In [1]:
import numpy as np
import pandas as pd
from collections import Counter
import warnings, os, gc, re
import matplotlib.pyplot as plt
import psutil
import swifter
# import icd10
from ast import literal_eval

# from plotnine import ggplot, aes, geom_line
from plotnine import *
# import pygal as pg

%matplotlib inline

warnings.filterwarnings('ignore')
pd.options.display.float_format = '{:.4f}'.format

In [ ]:
path, dirs, files = next(os.walk("chunks/"))
directory = 'chunks/'
file_count = len(files)
year_start = 2013
year_end = 2020

In [ ]:
def turnaround_time_1(dframe, col_name_grp, tab_name, tab_name_365):   
    df2 = df1.loc[df1['TAT'] >= 0].groupby([col_name_grp])['TAT'] \
            .agg(['mean', 'median', 'std', 'min', 'max', q25, q75, 'count']) 
    df3 = df1.loc[(df1['TAT'] > 60)].groupby([col_name_grp]).size().to_frame().reset_index()
    df3 = df3.set_axis([col_name_grp, 'numgreater60'], axis=1, inplace=False)
    df2 = df2.merge(df3, how='inner', on=col_name_grp)
    df2['%>60days'] = 100*(df2['numgreater60']/df2['count'])
    df2.to_excel(writer, sheet_name=tab_name, index=False)
    del[df2]
    del[df3]

    df2 = df1.loc[(df1['TAT'] >= 0) & (df1['TAT'] <= 365)].groupby([col_name_grp])['TAT'] \
            .agg(['mean', 'median', 'std', 'min', 'max', q25, q75, 'count']) 
    df3 = df1.loc[(df1['TAT'] > 60) & (df1['TAT'] <= 365)].groupby([col_name_grp]).size().to_frame().reset_index()
    df3 = df3.set_axis([col_name_grp, 'numgreater60'], axis=1, inplace=False)
    df2 = df2.merge(df3, how='inner', on=col_name_grp)
    df2['%>60days'] = 100*(df2['numgreater60']/df2['count'])
    df2.to_excel(writer, sheet_name=tab_name_365, index=False)
    del[df2]
    del[df3]
    
def unpaid_process(dframe, \
                check_dt_col, rr_col, amt_col, acr_amt_col, \
                agg_col, tab_name, writer1):
    df2 = dframe[dframe[check_dt_col].isna() & ~dframe[rr_col].isna()] 
    df2[amt_col].fillna(df2[acr_amt_col], inplace=True)    
    df2.groupby([agg_col])[amt_col].agg(['mean', 'median', 'count', 'min', 'max']) \
        .to_excel(writer1, sheet_name=tab_name, index=True)
    del[df2]

In [ ]:
for i in range(year_start, year_end+1):
    df1 = pd.read_csv(os.path.join(directory, str(i) + '_new_1.csv'), low_memory=False, index_col=0)

    # for lighter pandas df and faster execution
    df1['PRO_NAME'] = df1['PRO_NAME'].astype('category')
    df1['INSTITUTION_NAME'] = df1['INSTITUTION_NAME'].astype('category')
    df1['OWNERSHIP'] = df1['OWNERSHIP'].astype('category')
    df1['INSTITUTION_CLASS'] = df1['INSTITUTION_CLASS'].astype('category')
    df1['INSTITUTION_PROVINCE'] = df1['INSTITUTION_PROVINCE'].astype('category')
    df1['INSTITUTION_MUNICIPALITY'] = df1['INSTITUTION_MUNICIPALITY'].astype('category')
    df1['MEM_CATEGORY'] = df1['MEM_CATEGORY'].astype('category')
    df1['MEM_SUB_CATEGORY'] = df1['MEM_SUB_CATEGORY'].astype('category')

    # computing for Turnaround Time
    df1[['CHECK_DT','RECEIVED_REFILED_DATE']] = df1[['CHECK_DT','RECEIVED_REFILED_DATE']].apply(pd.to_datetime)
    df1['TAT'] = (df1['CHECK_DT'] - df1['RECEIVED_REFILED_DATE']).dt.days

    print(i)
    print(len(df1.index))
    total_year = len(df1.index)
    total_paid_amt = df1['PAID_AMOUNT'].sum()

    del[df1]
    gc.collect()
    gc.collect()